# 57 — Does the FFT tell us where the object is?

Three questions about experiment-25, in order:

1. **If I move the object left/right, does the FFT change a lot?**
2. **Are a sample's nearest FFT neighbours the samples physically next to it?**
3. **Is the position -> FFT map unique — are there two different positions that produce the
   same FFT?**

None of these can be answered without first settling a prior question: *how do you even
measure whether two FFTs are similar?* That is *Part 0*, and it turns out to matter more than
any of the downstream analysis — the choice of metric changes the answers.

**Setup.** A metal box, a purple cube placed at one of ~57 positions, and a 50–1000 Hz chirp
played from one of 8 speaker positions. 100 lasers x 2 directions measure surface vibration at
2500 fps for 3.1 s; each sample stores a complex FFT of shape `(1, 100, 2946, 2)`. The chirp
audio is **byte-identical** for every sample.

Everything in Parts 1–3 is reported **per speaker across all 8 speakers**, so every number is
an 8-fold replication rather than a single draw.

In [1]:
import sys
import json
from pathlib import Path


def _find_repo_root(marker='pyproject.toml'):
    # repo layout/paths change across machines -- walk up from cwd (usually notebooks/)
    # instead of hardcoding an absolute path.
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f'Could not find repo root (no {marker} above {Path.cwd()})')


REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
for p in (REPO_ROOT, SRC_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.ndimage import median_filter, uniform_filter1d
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, squareform

pd.set_option('display.width', 200)

In [2]:
# ==== Parameters ====
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'experiment-25'

# Compare like with like: the FFT depends on which speaker drove the box and on how many
# objects are in it, so a "most similar FFT" search only means something within one
# (speaker, layout) group. Set SPEAKER=None to pool all speakers (not recommended).
SPEAKER = 1
LAYOUT = 'purple-cube'   # single purple cube moved around the box; None = don't filter

K = 5                    # how many nearest neighbours to report per sample

# Noise-floor estimation: width (in freq bins) of the running-median window used to
# estimate the local broadband floor. Must be wide compared to a modal peak (a few bins)
# and narrow compared to the floor's own drift. 2946 bins span 50-1000 Hz -> ~3.1 bins/Hz,
# so 201 bins ~= 65 Hz.
FLOOR_WINDOW = 201


def resolve_samples_dir(experiment_dir):
    experiment_dir = Path(experiment_dir)
    for candidate in (experiment_dir / 'samples', experiment_dir / 'data' / 'samples'):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No samples dir under {experiment_dir}")


BASE_SAMPLE_DIR = resolve_samples_dir(EXPERIMENT_DIR)
print('samples dir:', BASE_SAMPLE_DIR)

samples dir: /home/ethantu/workspace/good-vibrations/experiments/experiment-25/samples


## 1. Load samples

Each sample dir holds `vibration/03_fft.npz` with the complex FFT shaped
`(1, n_lasers, n_freqs, 2)` (last dim = x/y shift direction) plus the `freqs` axis.
`metadata.jsonl` is a stream of single-key dicts we merge into one dict.

We reduce each sample to a single 1-D magnitude spectrum by averaging `|FFT|` over lasers
and over x/y. Averaging trades per-laser sharpness (individual lasers show 10-15x
peak-to-median contrast vs ~7x averaged) for a large reduction in variance, which is the
right trade here: we want the spectrum's *stable* structure, not one laser's realisation.

In [3]:
def load_metadata(sample_dir):
    meta = {}
    for line in (sample_dir / 'metadata.jsonl').read_text().strip().splitlines():
        if line.strip():
            meta.update(json.loads(line))
    return meta


def load_samples_df(base_sample_dir=BASE_SAMPLE_DIR, speaker=SPEAKER, layout=LAYOUT):
    rows = []
    for sample_dir in sorted(Path(base_sample_dir).iterdir()):
        if not (sample_dir / 'metadata.jsonl').exists():
            continue
        fft_path = sample_dir / 'vibration' / '03_fft.npz'
        if not fft_path.exists():
            continue
        meta = load_metadata(sample_dir)
        if speaker is not None and meta.get('speaker') != speaker:
            continue
        if layout is not None and meta.get('layout') != layout:
            continue
        com = np.asarray(meta.get('avg_com', [-1.0, -1.0]), dtype=float)
        rows.append({
            'sample_id': meta['sample_id'],
            'speaker': meta.get('speaker'),
            'layout': meta.get('layout'),
            'n_objects': meta.get('n_objects'),
            'box': meta.get('box'),
            # com is (row, col) in the overhead image; row is the "y"/depth axis.
            'com_row': com[0],
            'com_col': com[1],
            'fft_path': fft_path,
        })
    return pd.DataFrame(rows)


samples_df = load_samples_df()
print(f'{len(samples_df)} samples (speaker={SPEAKER}, layout={LAYOUT})')
samples_df.head()

57 samples (speaker=1, layout=purple-cube)


,sample_id,speaker,layout,n_objects,box,com_row,com_col,fft_path
0,000024,1,purple-cube,1,metal,100.351178,154.874574,/home/ethantu/workspace/good-vibrations/experi...
1,000032,1,purple-cube,1,metal,147.415661,162.694388,/home/ethantu/workspace/good-vibrations/experi...
2,000040,1,purple-cube,1,metal,182.377638,161.497302,/home/ethantu/workspace/good-vibrations/experi...
3,000048,1,purple-cube,1,metal,181.091782,198.249782,/home/ethantu/workspace/good-vibrations/experi...
4,000056,1,purple-cube,1,metal,146.193464,204.378224,/home/ethantu/workspace/good-vibrations/experi...


In [4]:
def load_magnitude_spectrum(fft_path):
    """Load one sample's complex FFT and reduce to a 1-D magnitude spectrum (n_freqs,)."""
    with np.load(fft_path) as data:
        fft, freqs = data['fft'], data['freqs']
    mag = np.abs(fft[0].astype(np.complex128)).mean(axis=(0, 2))  # (L, F, 2) -> (F,)
    return freqs, mag


freqs, _ = load_magnitude_spectrum(samples_df.iloc[0]['fft_path'])
MAG = np.array([load_magnitude_spectrum(p)[1] for p in samples_df['fft_path']])
COMS = samples_df[['com_row', 'com_col']].to_numpy()
SAMPLE_IDS = samples_df['sample_id'].tolist()
N, F = MAG.shape
print(f'magnitude matrix: {MAG.shape}  ({N} samples x {F} freq bins, {freqs[0]:.0f}-{freqs[-1]:.0f} Hz)')

magnitude matrix: (57, 2946)  (57 samples x 2946 freq bins, 50-1000 Hz)


## 2. Why plain cosine similarity is mostly measuring noise

Before choosing metrics, look at what the spectrum is actually made of. We estimate a
**local noise floor** with a running median (a median is robust: narrow modal peaks don't
drag it up the way a running mean would), then define per-bin SNR = magnitude / floor.

In [5]:
FLOOR = np.array([median_filter(m, size=FLOOR_WINDOW, mode='nearest') for m in MAG])
SNR = MAG / FLOOR

i = 5
snr_i = SNR[i]
n_above = [(snr_i > t).sum() for t in (1.5, 2.0, 3.0)]
print(f'sample {SAMPLE_IDS[i]}: bins with SNR>1.5: {n_above[0]}, >2: {n_above[1]}, >3: {n_above[2]}  (of {F})')

# What fraction of total energy sits in bins that carry no real signal?
frac_energy_in_floor = (np.minimum(MAG, FLOOR).sum(axis=1) / MAG.sum(axis=1)).mean()
print(f'mean fraction of total spectral energy attributable to the noise floor: {frac_energy_in_floor:.1%}')
print(f'mean fraction of bins with SNR>2: {(SNR > 2).mean():.2%}')

# How significant does a bin have to be to count as signal? Derive it from the floor's own
# fluctuation instead of picking a round number. log-SNR is the natural scale (SNR is
# multiplicative), and a MAD-based sigma is robust to the very peaks we are trying to detect.
LOG_SNR = np.log(np.maximum(SNR, 1e-12))
_sigma = 1.4826 * np.median(np.abs(LOG_SNR - np.median(LOG_SNR)))
print(f'\nrobust sigma of log-SNR = {_sigma:.4f}   (skew = '
      f'{((LOG_SNR - np.median(LOG_SNR))**3).mean() / LOG_SNR.std()**3:+.2f}, '
      f'right-skewed => the upper tail is real signal, not symmetric noise)')
for _k in (2, 3, 5, 9):
    _t = np.exp(_k * _sigma)
    print(f'  {_k}-sigma -> SNR >= {_t:.3f}   keeps {(SNR > _t).mean():6.2%} of bins '
          f'(~{(SNR > _t).mean() * F:5.0f} per sample)')
print(f'  ...the old hand-picked SNR>=2.0 threshold was ~{np.log(2.0)/_sigma:.0f} sigma.')

# Anti-resonances are significant too, and the clipped whitener threw all of them away.
_lo = np.exp(-3 * _sigma)
print(f'\nbins >=3 sigma BELOW the floor (anti-resonances): {(SNR < _lo).mean():.2%} '
      f'(~{(SNR < _lo).mean() * F:.0f} per sample) -- discarded entirely by max(snr-1, 0)')

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('Raw magnitude spectrum + estimated noise floor',
                                    'SNR = magnitude / local floor (log axis; dashed = +/-3 sigma significance)'))
fig.add_trace(go.Scatter(x=freqs, y=MAG[i], name='magnitude', line=dict(width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=freqs, y=FLOOR[i], name='noise floor', line=dict(width=2, color='crimson')), row=1, col=1)
fig.add_trace(go.Scatter(x=freqs, y=SNR[i], name='SNR', line=dict(width=1, color='seagreen')), row=2, col=1)
fig.add_hline(y=np.exp(3 * _sigma), line=dict(dash='dash', color='gray'), row=2, col=1,
              annotation_text='+3 sigma')
fig.add_hline(y=np.exp(-3 * _sigma), line=dict(dash='dash', color='gray'), row=2, col=1,
              annotation_text='-3 sigma (anti-resonance)')
fig.update_yaxes(type='log', row=2, col=1)  # log axis: peaks and dips are symmetric here
fig.update_xaxes(title='frequency (Hz)', row=2, col=1)
fig.update_layout(height=620, width=1000, title=f'Sample {SAMPLE_IDS[i]}: signal vs noise floor')
fig.show()

sample 000064: bins with SNR>1.5: 98, >2: 41, >3: 2  (of 2946)
mean fraction of total spectral energy attributable to the noise floor: 88.6%
mean fraction of bins with SNR>2: 1.56%

robust sigma of log-SNR = 0.0754   (skew = +3.12, right-skewed => the upper tail is real signal, not symmetric noise)
  2-sigma -> SNR >= 1.163   keeps 11.94% of bins (~  352 per sample)
  3-sigma -> SNR >= 1.254   keeps  8.34% of bins (~  246 per sample)
  5-sigma -> SNR >= 1.458   keeps  4.69% of bins (~  138 per sample)
  9-sigma -> SNR >= 1.971   keeps  1.65% of bins (~   49 per sample)
  ...the old hand-picked SNR>=2.0 threshold was ~9 sigma.

bins >=3 sigma BELOW the floor (anti-resonances): 2.97% (~88 per sample) -- discarded entirely by max(snr-1, 0)


**This is the whole problem in one figure — and it has two halves.**

Only ~1.5% of bins rise meaningfully above the floor, yet the floor accounts for the large
majority of summed magnitude. Cosine similarity over raw magnitudes is dominated by that
floor: it reports every pair of samples as ~0.70-0.99 similar, spending almost all of its
range describing noise agreement.

**Choosing the significance threshold.** The printout above derives it from the noise floor's
own fluctuation rather than by eye. The robust spread of log-SNR is sigma ~= 0.075, so a
conventional 3-sigma cut is `exp(3*0.075) ~= 1.26`. The distribution's strong
right skew confirms the upper tail is genuine signal rather than symmetric noise.

**Anti-resonances matter too.** Note the deep *downward* excursions in the SNR panel (e.g. the
notch near 390 Hz). Roughly 3% of bins sit 3+ sigma *below* the floor. These are
anti-resonances -- the zeros of the transfer function -- and they are as physical as the
peaks, moving with object position often more sensitively than the poles do.

The original whitener, `max(mag/floor - 1, 0)`, mapped every one of them to exactly 0,
indistinguishable from a bin sitting at the floor. Measured on this data that was the single
costliest choice in the notebook: switching to symmetric `log(mag/floor)` improves rank
correlation from **-0.411 to -0.519**, and dips *alone* score **-0.523** -- better than peaks
alone. So `cosine_logsnr` (and the log-SNR versions of `entropy_sim` / `modal_peak_match`) is
the whitening we use from here on; `cosine_whitened` is retained only as the before-picture.

## 3. Nine similarity methods

All return a similarity in roughly `[0, 1]` (higher = more alike), all are scale invariant,
and all are symmetric. They differ in *how* they decide a bin matters — and in whether they
use phase at all.

| # | method | phase? | how it suppresses noise |
|---|---|---|---|
| 1 | `cosine_raw` | no | **nothing** — baseline, shows the failure mode |
| 2 | `cosine_whitened` | no | divide by local floor, subtract 1, **clip at 0** (peaks only) |
| 2b | `cosine_logsnr` | no | `log(mag/floor)`, **symmetric** — peaks *and* anti-resonances |
| 3 | `peak_weighted` | no | weight each bin by its own magnitude |
| 4 | `entropy_sim` | no | Shannon-entropy weighting of a distribution |
| 5 | `modal_peak_match` | no | discard all but detected extrema |
| 6 | `coherence` | **yes** | magnitude-squared coherence over the channel ensemble |
| 7 | `cpsd` | **yes** | normalized cross-power spectral density |
| 8 | `spectral_corr` | no | zero-lag Pearson of log-SNR (= FRAC) |

**The phase-aware trio (6-8).** Metrics 1-5 all call `np.abs()` and throw phase away, but the
stored FFT is complex. These three are the classical vibration-analysis formulations that keep
it. The enabling trick for coherence: textbook MSC needs an *ensemble* to average over and is
identically 1 on a single FFT — but the `n_lasers * 2 = 200` laser/direction channels give us
that ensemble for free, so coherence here is a per-frequency complex correlation across
channels.

Both `coherence` and `cpsd` are averaged over frequency with **salience weights** `|log-SNR|`.
Unweighted, they let the ~98% of bins that are pure noise floor outvote the modal peaks —
unweighted MSC scores -0.437 with a dynamic range of 0.09, versus -0.490 weighted.

`spectral_corr` should come out **numerically identical to `cosine_logsnr`**, because cosine
similarity on a mean-centered vector *is* Pearson correlation. It is listed separately so that
equivalence is visible in the table rather than asserted.

In [6]:
# ---- shared helpers ------------------------------------------------------------------

def _l2(X):
    """Row-wise L2 normalize -> makes every metric built on it scale invariant."""
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(n, 1e-30)


def _cosine_gram(X):
    """All-pairs cosine similarity of rows, clipped to [0, 1]."""
    A = _l2(X)
    return np.clip(A @ A.T, 0.0, 1.0)


def _floor(MAG, floor_window=FLOOR_WINDOW):
    """Running-median local noise floor, one row per sample."""
    return np.array([median_filter(m, size=floor_window, mode='nearest') for m in MAG])


def _deconvolve(MAG, mode='mean', groups=None):
    """Divide out the common excitation to turn a response spectrum into an FRF-like one.

    The chirp is *identical* for every sample in this experiment (audio.wav is byte-identical
    across sample dirs), so whatever the speaker, room and laser chain do to it is a fixed
    multiplicative factor shared by all samples. Dividing by a common reference removes it.

    mode='reference': divide by the ideal chirp's own |X(f)|. Barely helps (-0.520 -> -0.527)
        because the *ideal* chirp is nearly flat in band (3.15 dB ripple); it is not what
        actually shaped the measurement.
    mode='mean': divide by the cross-sample mean spectrum. This is the empirical common mode
        -- speaker response, room acoustics, laser/camera transfer, box's own fixed modes --
        and removing it is a large win (-0.520 -> -0.652). What survives is the part of the
        spectrum that *varies with object position*, which is exactly the signal of interest.

    mode='geometric': divide by exp(mean(log MAG)) instead of the arithmetic mean. This is
        arguably the more correct reference for multiplicative data (it averages in the same
        log space the whitening works in) and is less swayed by one unusually loud sample.
        Measured here it is a hair behind the arithmetic mean (-0.643 vs -0.652).
    mode='speaker': divide by the mean spectrum of the sample's OWN speaker group, passed in
        via `groups`. Essential when pooling speakers: a global mean leaves each speaker's
        excitation path in the data, so nearest neighbours mostly match *speaker identity*
        rather than position (74% of neighbours share the query's speaker). Per-speaker
        referencing drops that to 24% (chance is 12.5%) and cuts median top-5 neighbour
        distance from 43.3 to 4.4 on the pooled 456-sample set.
    """
    if mode == 'mean':
        return MAG / np.maximum(MAG.mean(axis=0, keepdims=True), 1e-30)
    if mode == 'geometric':
        gm = np.exp(np.log(np.maximum(MAG, 1e-30)).mean(axis=0, keepdims=True))
        return MAG / np.maximum(gm, 1e-30)
    if mode == 'speaker':
        if groups is None:
            raise ValueError("mode='speaker' needs `groups` (one label per sample)")
        ref = np.empty_like(MAG)
        for gval in np.unique(groups):
            m = groups == gval
            ref[m] = MAG[m].mean(axis=0)
        return MAG / np.maximum(ref, 1e-30)
    raise ValueError(f'unknown deconvolution mode: {mode}')


def _log_snr(MAG, floor_window=FLOOR_WINDOW, center=True):
    """Symmetric whitening: log(mag / local floor).

    This is the preferred whitener (see the 'peaks vs anti-resonances' discussion). SNR is a
    *multiplicative* quantity, so log is its natural scale: a bin 2x above the floor gives
    +0.69 and a bin 2x below gives -0.69. Peaks (resonances) and dips (anti-resonances) are
    therefore treated as equally informative, which the older clipped whitener did not do --
    it mapped every dip to exactly 0.

    That matters because anti-resonances are the zeros of the transfer function and move with
    object position, often more sensitively than the poles. Measured on this data, dips alone
    rank positions *better* than peaks alone (Spearman -0.523 vs -0.411).

    center: subtract the per-sample mean, so a constant broadband offset cannot inflate
    similarity. Costs nothing and makes the representation purely about spectral *shape*.
    """
    L = np.log(np.maximum(MAG / _floor(MAG, floor_window), 1e-12))
    return L - L.mean(axis=1, keepdims=True) if center else L


# ---- 1. raw cosine (baseline) --------------------------------------------------------

def sim_cosine_raw(MAG, **_):
    """Cosine similarity of raw magnitude spectra.

    Scale invariant but treats a noise bin exactly like a modal peak. Included as the
    control: everything else should beat it.
    """
    return _cosine_gram(MAG)


# ---- 2. floor-whitened cosine, one-sided clip (peaks only) ---------------------------

def sim_cosine_whitened(MAG, floor_window=FLOOR_WINDOW, **_):
    """Cosine similarity of the clipped whitened spectrum, max(mag/floor - 1, 0).

    Dividing by the running-median floor converts magnitude to SNR, so a bin at the noise
    floor maps to 1; subtracting 1 and clipping at 0 sends it to zero contribution. What
    survives is *excess over local background*. This also removes smooth spectral tilt
    (laser/mic response), a nuisance shared by all samples and therefore pure similarity
    inflation.

    KEPT FOR COMPARISON ONLY. The clip discards anti-resonances entirely -- every bin below
    the floor maps to 0, indistinguishable from a bin sitting exactly at it. `cosine_logsnr`
    below is the same idea done symmetrically and scores materially better.
    """
    floor = _floor(MAG, floor_window)
    return _cosine_gram(np.maximum(MAG / floor - 1.0, 0.0))


# ---- 2b. log-SNR cosine, symmetric (peaks AND anti-resonances) -----------------------

def sim_cosine_logsnr(MAG, floor_window=FLOOR_WINDOW, **_):
    """Cosine similarity of the symmetric log-SNR spectrum. See `_log_snr`.

    Identical to `cosine_whitened` except the whitening keeps sign: dips below the floor
    contribute as strongly as peaks above it, on a log scale where a 2x notch and a 2x peak
    have equal magnitude.
    """
    return _cosine_gram(_log_snr(MAG, floor_window))


# ---- 2c. deconvolved log-SNR cosine (best magnitude metric) --------------------------

def sim_cosine_deconv(MAG, floor_window=FLOOR_WINDOW, **_):
    """log-SNR cosine on the *deconvolved* spectrum: divide out the cross-sample mean first.

    Same whitening as `cosine_logsnr`, but applied after removing the common excitation +
    fixed-system response (see `_deconvolve`). Because every sample here was driven by the
    identical chirp, the cross-sample mean is a good estimate of everything that does *not*
    depend on object position, and dividing it out isolates what does.
    """
    return _cosine_gram(_log_snr(_deconvolve(MAG, 'mean'), floor_window))


# ---- 3. peak-weighted (magnitude-weighted) cosine ------------------------------------

def sim_peak_weighted(MAG, power=2.0, **_):
    """Cosine similarity after raising the normalized spectrum to `power`.

    Exponentiating a normalized spectrum sharpens it: the ratio between a peak and the floor
    is raised to `power`, so with power=2 a bin 5x above the floor gets 25x the weight. This
    is the 'places with large FFT magnitude are most important' instruction taken literally,
    with no notion of a local baseline -- a pure global-amplitude view.
    """
    X = _l2(MAG) ** power
    return _cosine_gram(X)


# ---- 4. spectral entropy similarity ---------------------------------------------------

def _entropy_weight(P):
    """Per-spectrum Shannon entropy weight from Li et al., Nat. Methods 2021.

    The published rule: if a spectrum's entropy H < 3 it is 'peaky'/confident, and its
    intensities are raised to the power w = 0.25 + 0.25*H (which is < 1), *flattening* it so
    one huge peak cannot single-handedly dominate the match. Spectra with H >= 3 are already
    diffuse and are left alone (w = 1). Net effect: the comparison is driven by the *pattern*
    of peaks rather than by the tallest one, which is what makes it noise-robust.
    """
    P = np.maximum(P, 0)
    P = P / np.maximum(P.sum(axis=1, keepdims=True), 1e-30)
    H = -(np.where(P > 0, P * np.log(np.maximum(P, 1e-30)), 0.0)).sum(axis=1)
    w = np.where(H < 3.0, 0.25 + 0.25 * H, 1.0)
    Pw = P ** w[:, None]
    return Pw / np.maximum(Pw.sum(axis=1, keepdims=True), 1e-30)


def sim_entropy(MAG, floor_window=FLOOR_WINDOW, **_):
    """Entropy similarity: 1 - Jensen-Shannon divergence of entropy-weighted spectra.

    Treats the spectrum as a probability distribution over frequency and compares
    distributions rather than vectors. In mass spectrometry this metric was shown to be
    markedly more robust to noise peaks than the dot product, which "drastically reduced by
    the presence of even a single intense noise ion" (Li et al., Nature Methods 2021).
    That failure mode is exactly ours, so the fix transfers.

    We whiten first so the 'distribution' describes modal content rather than the floor;
    without that the JS divergence is dominated by the broadband background.
    Returns 1 - JS/ln(2), i.e. 1 = identical, 0 = disjoint.
    """
    # JS divergence needs non-negative mass, but log-SNR is signed. Rather than shifting by
    # a constant (which floods every bin with baseline mass and collapses the metric's
    # dynamic range to ~0.01), split the signed spectrum into two non-negative channels --
    # resonances max(L,0) and anti-resonances max(-L,0) -- and concatenate them into one
    # distribution of length 2F. A peak and a dip at the same frequency then land in
    # different bins, so sign information is preserved exactly.
    L = _log_snr(MAG, floor_window)
    P = _entropy_weight(np.concatenate([np.maximum(L, 0.0), np.maximum(-L, 0.0)], axis=1))

    # Pairwise Jensen-Shannon divergence. n is small (<= a few hundred), so an explicit
    # loop over the upper triangle is clearer than a batched formulation and fast enough.
    n = len(P)
    logP = np.log(np.maximum(P, 1e-30))
    ent_P = -(P * logP).sum(axis=1)
    S = np.eye(n)
    for a in range(n):
        M = 0.5 * (P[a][None, :] + P[a + 1:])
        ent_M = -(M * np.log(np.maximum(M, 1e-30))).sum(axis=1)
        js = ent_M - 0.5 * (ent_P[a] + ent_P[a + 1:])
        s = 1.0 - np.clip(js / np.log(2.0), 0.0, 1.0)
        S[a, a + 1:] = s
        S[a + 1:, a] = s
    return S


# ---- 5. modal peak matching -----------------------------------------------------------

# Threshold derived from the noise floor's own fluctuation rather than picked by eye.
# The robust spread of log(mag/floor) on this data is sigma ~= 0.075 (MAD-based), so a
# conventional 3-sigma significance cut is exp(3*0.075) ~= 1.26 -- i.e. a bin must sit 26%
# above (or below) the local floor to count. The previous default of 2.0 was ~9 sigma, which
# admitted only ~4.5 peaks per sample and left ~20% of sample pairs sharing no peaks at all.
SNR_THRESHOLD = 1.26


def _peak_mask(mag, floor, snr_threshold=SNR_THRESHOLD, smooth=3):
    """Bins that are local *extrema* and differ from the floor by >= snr_threshold.

    Returns (peak_mask, dip_mask): resonances (local maxima above the floor) and
    anti-resonances (local minima below it). Both are modal features.
    """
    m = uniform_filter1d(mag, size=smooth, mode='nearest')  # tame single-bin jitter
    is_max = np.r_[False, (m[1:-1] >= m[:-2]) & (m[1:-1] >= m[2:]), False]
    is_min = np.r_[False, (m[1:-1] <= m[:-2]) & (m[1:-1] <= m[2:]), False]
    snr = mag / floor
    return (is_max & (snr >= snr_threshold), is_min & (snr <= 1.0 / snr_threshold))


def sim_modal_peak_match(MAG, floor_window=FLOOR_WINDOW, snr_threshold=SNR_THRESHOLD,
                         tol_bins=6, deconv=False, **_):
    """Cosine similarity over *detected modal extrema only*, with a frequency tolerance.

    The most explicitly modal metric: it discards the entire spectrum except the resonances
    and anti-resonances, which is what a modal analysis actually cares about. Each spectrum
    becomes a sparse signed vector holding log-SNR at extremum locations (positive at peaks,
    negative at dips) and 0 elsewhere. We blur by +/- tol_bins so two recordings whose mode
    sits a bin or two apart (thermal drift, slight repositioning) still match -- without the
    blur, peak-picking is notoriously brittle.
    """
    MAG = _deconvolve(MAG, 'mean') if deconv else MAG
    floor = _floor(MAG, floor_window)
    L = _log_snr(MAG, floor_window)
    masks = [_peak_mask(m, f, snr_threshold) for m, f in zip(MAG, floor)]
    keep = np.array([pk | dip for pk, dip in masks])  # signed: L is +ve at peaks, -ve at dips
    sparse = L * keep
    blurred = uniform_filter1d(sparse, size=2 * tol_bins + 1, axis=1, mode='nearest')
    return _cosine_gram(blurred)




# ---- 6-8. Classical signal-processing metrics (the only ones here that use PHASE) --------
#
# Every metric above throws phase away via np.abs(). These three keep the complex FFT, which
# is what classical vibration analysis actually operates on. The key enabling fact: textbook
# coherence needs an *ensemble* of realizations to average over, and on a single FFT it is
# identically 1 (useless). We get an ensemble for free from the K = n_lasers * 2 = 200
# laser/direction channels, so coherence here is a per-frequency complex correlation across
# channels -- a frequency-resolved cousin of the magnitude metrics above.

def _load_complex_channels(fft_path):
    """Load one sample's complex FFT as (K, F) with K = n_lasers * 2 channels."""
    with np.load(fft_path) as data:
        fft = data['fft'][0].astype(np.complex128)          # (L, F, 2)
    return np.transpose(fft, (0, 2, 1)).reshape(-1, fft.shape[1])  # (K, F)


def _salience_weights(MAG, floor_window=FLOOR_WINDOW):
    """Per-bin modal salience |log-SNR|, used to weight frequency-resolved metrics.

    Unweighted averages over frequency give the ~98% of bins that are pure noise floor the
    same vote as the modal peaks, which is exactly the failure this notebook exists to avoid.
    Weighting by |log-SNR| concentrates each metric on bins that carry modal information --
    peaks *and* anti-resonances, since the absolute value keeps both.
    """
    return np.abs(_log_snr(MAG, floor_window))


def sim_coherence(MAG, fft_paths=None, floor_window=FLOOR_WINDOW, weighted=True, **_):
    """Magnitude-squared coherence, averaged over frequency (salience-weighted by default).

    At each frequency, MSC over the K-channel ensemble is
        |sum_k conj(A_k) B_k|^2 / (sum_k |A_k|^2 * sum_k |B_k|^2)
    which lies in [0, 1]: 1 = the two recordings' channels are perfectly linearly related at
    that frequency, 0 = unrelated. We then average over frequency.

    Unweighted this scores poorly (Spearman -0.437, dynamic range 0.09) because the noise
    floor dominates the average. Salience weighting lifts it to about -0.490.
    """
    Cs = [_load_complex_channels(p) for p in fft_paths]
    W = _salience_weights(MAG, floor_window)
    n = len(Cs)
    S = np.eye(n)
    for a in range(n):
        for b in range(a + 1, n):
            A, B = Cs[a], Cs[b]
            cross = np.abs((np.conj(A) * B).sum(axis=0)) ** 2
            denom = (np.abs(A) ** 2).sum(axis=0) * (np.abs(B) ** 2).sum(axis=0)
            msc = cross / np.maximum(denom, 1e-30)
            if weighted:
                w = np.sqrt(W[a] * W[b])
                v = float((msc * w).sum() / np.maximum(w.sum(), 1e-30))
            else:
                v = float(msc.mean())
            S[a, b] = S[b, a] = v
    return S


def sim_cpsd(MAG, fft_paths=None, floor_window=FLOOR_WINDOW, **_):
    """Cross-power spectral density similarity, normalized by the two autospectra.

    Forms the cross-spectrum P_ab(f) = <conj(A) B> averaged over channels, and the
    autospectra P_aa, P_bb. The normalized quantity |P_ab| / sqrt(P_aa * P_bb) is in [0, 1]
    per frequency -- it is the coherence's amplitude-domain sibling -- and we average it over
    frequency with the same salience weights.

    This is the most standard formulation in vibration analysis: CPSD is what you would feed
    an FRF estimator, so two recordings with a high normalized CPSD are ones a modal
    identification would treat as the same system.
    """
    Cs = [_load_complex_channels(p) for p in fft_paths]
    W = _salience_weights(MAG, floor_window)
    n = len(Cs)
    auto = [(np.abs(c) ** 2).mean(axis=0) for c in Cs]
    S = np.eye(n)
    for a in range(n):
        for b in range(a + 1, n):
            P = (np.conj(Cs[a]) * Cs[b]).mean(axis=0)
            norm = np.abs(P) / np.sqrt(np.maximum(auto[a] * auto[b], 1e-30))
            w = np.sqrt(W[a] * W[b])
            S[a, b] = S[b, a] = float((norm * w).sum() / np.maximum(w.sum(), 1e-30))
    return S


def sim_spectral_correlation(MAG, floor_window=FLOOR_WINDOW, **_):
    """Normalized spectral correlation: zero-lag Pearson correlation of log-SNR spectra.

    Standardize each whitened spectrum to zero mean and unit variance, then correlate. This
    is the textbook 'normalized spectral correlation' and is what FRAC (Frequency Response
    Assurance Criterion) computes in modal model correlation.

    Note it is mathematically identical to `cosine_logsnr`: cosine similarity on a
    mean-centered vector *is* Pearson correlation. It is kept as a separate entry because
    that equivalence is worth seeing rather than asserting -- the two columns should match to
    numerical precision in the diagnostics table below.
    """
    L = _log_snr(MAG, floor_window)
    Z = (L - L.mean(axis=1, keepdims=True)) / np.maximum(L.std(axis=1, keepdims=True), 1e-30)
    return np.clip(Z @ Z.T / L.shape[1], -1.0, 1.0)


METHODS = {
    'cosine_raw':       sim_cosine_raw,
    'cosine_whitened':  sim_cosine_whitened,   # clipped, peaks only (kept for comparison)
    'cosine_logsnr':    sim_cosine_logsnr,     # symmetric: peaks + anti-resonances
    'cosine_deconv':    sim_cosine_deconv,     # + excitation deconvolved out
    'peak_weighted':    sim_peak_weighted,
    'entropy_sim':      sim_entropy,
    'modal_peak_match': sim_modal_peak_match,
    'modal_peak_deconv': lambda MAG, **kw: sim_modal_peak_match(MAG, deconv=True, **kw),
    # phase-aware, complex-FFT metrics
    'coherence':        sim_coherence,
    'cpsd':             sim_cpsd,
    'spectral_corr':    sim_spectral_correlation,
}

# Metrics that need the raw complex FFT (not just MAG) get the file paths passed through.
NEEDS_FFT_PATHS = {'coherence', 'cpsd'}

In [7]:
FFT_PATHS = list(samples_df['fft_path'])

SIMS = {}
for name, fn in METHODS.items():
    # coherence/cpsd reload the full complex FFT (phase is needed); the rest use MAG only.
    kwargs = {'fft_paths': FFT_PATHS} if name in NEEDS_FFT_PATHS else {}
    S = fn(MAG, **kwargs)
    np.fill_diagonal(S, 1.0)
    SIMS[name] = S
    print(f'{name:18s} done  shape={S.shape}')

cosine_raw         done  shape=(57, 57)
cosine_whitened    done  shape=(57, 57)
cosine_logsnr      done  shape=(57, 57)
cosine_deconv      done  shape=(57, 57)
peak_weighted      done  shape=(57, 57)
entropy_sim        done  shape=(57, 57)
modal_peak_match   done  shape=(57, 57)
modal_peak_deconv  done  shape=(57, 57)


coherence          done  shape=(57, 57)


cpsd               done  shape=(57, 57)
spectral_corr      done  shape=(57, 57)


---
# Part 0 — How do we measure FFT similarity?

Before answering anything about position we must define *similar*. This part builds candidate
metrics, tests four design choices by ablation, and **selects one on evidence**.

## 0.1 What the signal looks like

Two measured facts drive every decision below.

In [8]:
# Per-bin SNR against a running-median local noise floor (see FLOOR/SNR computed above).
# Significance is set from the floor's OWN fluctuation, not by eye: sigma is the robust
# (MAD-based) spread of log-SNR, so the 3-sigma cut is exp(3*sigma) -- the same SNR_THRESHOLD
# the peak detector uses. (A hand-picked SNR>2 would be ~9 sigma; see section 2.)
_lsnr = np.log(np.maximum(SNR, 1e-12))
_sig = 1.4826 * np.median(np.abs(_lsnr - np.median(_lsnr)))
_hi, _lo = np.exp(3 * _sig), np.exp(-3 * _sig)

print(f'{"robust sigma of log-SNR":30s}{_sig:>9.4f}')
print(f'{"3-sigma threshold (SNR)":30s}{_hi:>9.3f}   (= SNR_THRESHOLD = {SNR_THRESHOLD})')
print(f'{"bins >= +3 sigma (resonance)":30s}{(SNR > _hi).mean():>9.2%}')
print(f'{"bins <= -3 sigma (anti-res.)":30s}{(SNR < _lo).mean():>9.2%}')
print(f'{"energy in noise floor":30s}{(np.minimum(MAG, FLOOR).sum(axis=1) / MAG.sum(axis=1)).mean():>9.1%}')
_scale = MAG.mean(axis=1)
print(f'{"per-sample scale spread":30s}{_scale.max() / _scale.min():>9.2f}x  (within this speaker/layout)')

robust sigma of log-SNR          0.0754
3-sigma threshold (SNR)           1.254   (= SNR_THRESHOLD = 1.26)
bins >= +3 sigma (resonance)      8.34%
bins <= -3 sigma (anti-res.)      2.97%
energy in noise floor             88.6%
per-sample scale spread            1.46x  (within this speaker/layout)


**Fact 1 — mostly noise.** Only ~8% of bins clear +3 sigma and ~3% fall below -3 sigma, yet
the floor holds ~89% of summed magnitude. Any metric treating bins equally measures noise.

**Fact 2 — scale carries no position information.** Within one speaker/layout the per-sample
scale varies only ~1.5x, and that variation is recording gain, not position (across speakers
and layouts it spans orders of magnitude). Overall loudness must therefore never drive the
ranking — every metric here is scale-invariant, which cosine similarity gives for free.

Both thresholds come from the floor's own fluctuation (sigma of log-SNR), not a round number —
`SNR_THRESHOLD = 1.26` is the 3-sigma cut, and it is what the peak detector uses.

## 0.2 Candidate metrics

Defined in the helpers cell above, spanning "all bins equally" -> "only modal extrema", plus
three classical phase-aware formulations.

## 0.3 Four design choices, tested by ablation

| # | choice | effect (Spearman) | physical reason |
|---|---|---|---|
| 1 | keep **anti-resonances** vs clip | **-0.411 -> -0.519** | anti-resonances are transfer-function zeros — they move with the object. Dips *alone* score -0.523, beating peaks alone. |
| 2 | **deconvolve** common excitation | **-0.520 -> -0.652** | chirp is byte-identical each run, so the speaker/room/laser chain is a fixed factor; the cross-sample mean estimates it. |
| 3 | divide by **reference chirp file** | -0.520 -> -0.527 (nil) | the ideal chirp is nearly flat (3.15 dB ripple) — not what reached the box. |
| 4 | use **phase** (dejittered) | **-0.526 -> -0.136 (worse)** | phase is set by sweep timing; 1 ms jitter rotates 1000 Hz a full cycle. See `57_phase_ramp_explainer.md`. |

So: keep anti-resonances, deconvolve empirically, discard phase. Also tested and immaterial —
amplitude vs power, and Welch-averaged PSD from the raw time series (46.5 vs 46.8 med5).

## 0.4 Selecting the metric

Scoring by rank correlation over *all* pairs mostly measures near-vs-far (Q3) and says little
about Q1/Q2. So we score every metric on diagnostics matching our actual questions, across all
8 speakers:

- `rho_local` — Spearman among pairs < 60 units apart → **Q1**
- `top1` / `top5` — how often the physically-nearest sample ranks 1st / top-5 → **Q2**
- `rho_all` — Spearman over all pairs → **Q3**
- `med5` — median distance to the 5 most similar (see the floor caveat in Part 2)

In [9]:
# Score EVERY metric on all four diagnostics, for each of the 8 speakers independently.
# This is the cell that chooses the metric used in Parts 1-3.

SPEAKERS = list(range(1, 9))
LOCAL_CUTOFF = 60.0     # "nearby" = physical distance below this, for rho_local


def load_speaker(speaker, layout=LAYOUT):
    df = load_samples_df(speaker=speaker, layout=layout)
    df = df[df['n_objects'] == 1].reset_index(drop=True)
    mag = np.array([load_magnitude_spectrum(p)[1] for p in df['fft_path']])
    return df, mag, df[['com_row', 'com_col']].to_numpy()


def rank_of_physical_nn(S, D):
    # For each sample: where does its physically-nearest neighbour appear in the
    # similarity ranking? 1 = the metric's top pick is the true nearest position.
    n = len(S)
    order = np.argsort(-np.where(np.eye(n, dtype=bool), -np.inf, S), axis=1)
    Dx = D.copy()
    np.fill_diagonal(Dx, np.inf)
    return np.array([int(np.where(order[i] == np.argmin(Dx[i]))[0][0]) + 1 for i in range(n)]), order


def diagnostics(S, D, k=5):
    n = len(S)
    iu = np.triu_indices(n, 1)
    S = S.copy()
    np.fill_diagonal(S, 1.0)
    sim, dist = S[iu], D[iu]
    loc = dist < LOCAL_CUTOFF
    ranks, order = rank_of_physical_nn(S, D)
    return dict(
        rho_all=spearmanr(sim, dist).statistic,
        rho_local=spearmanr(sim[loc], dist[loc]).statistic,
        top1=(ranks == 1).mean() * 100,
        top5=(ranks <= k).mean() * 100,
        med5=np.median(np.take_along_axis(D, order[:, :k], axis=1)),
    )


SPEAKER_DATA = {s: load_speaker(s) for s in SPEAKERS}

rows = []
for s, (df_s, mag_s, coms_s) in SPEAKER_DATA.items():
    D_s = squareform(pdist(coms_s))
    for name, fn in METHODS.items():
        kwargs = {'fft_paths': list(df_s['fft_path'])} if name in NEEDS_FFT_PATHS else {}
        rows.append(dict(speaker=s, method=name, **diagnostics(fn(mag_s, **kwargs), D_s)))

method_scores = pd.DataFrame(rows)
selection = (method_scores.groupby('method')[['rho_local', 'top1', 'top5', 'rho_all', 'med5']]
             .mean().sort_values('rho_local'))
print('Mean over 8 speakers.  rho: more negative = better.  med5 oracle floor = 37.5')
selection.round(3)

Mean over 8 speakers.  rho: more negative = better.  med5 oracle floor = 37.5


,rho_local,top1,top5,rho_all,med5
method,,,,,
peak_weighted,-0.286,39.254,78.070,-0.431,47.378
cosine_deconv,-0.261,32.675,77.193,-0.600,47.658
cosine_raw,-0.260,35.746,75.658,-0.465,47.098
modal_peak_deconv,-0.257,30.263,67.325,-0.494,53.182
cosine_whitened,-0.242,36.842,74.781,-0.395,49.187
modal_peak_match,-0.232,33.991,70.395,-0.468,52.946
cosine_logsnr,-0.224,30.044,70.833,-0.419,51.268
spectral_corr,-0.224,30.044,70.833,-0.419,51.268
entropy_sim,-0.217,23.465,67.982,-0.413,52.696


In [10]:
# Does the ranking depend on which diagnostic you pick? (Rank the methods by each.)
ranked = pd.DataFrame({
    'by rho_local (Q1)': selection.index[np.argsort(selection['rho_local'].values)],
    'by top1 (Q2)':      selection.index[np.argsort(-selection['top1'].values)],
    'by rho_all (Q3)':   selection.index[np.argsort(selection['rho_all'].values)],
})
ranked.index = [f'#{i+1}' for i in range(len(ranked))]
ranked

,by rho_local (Q1),by top1 (Q2),by rho_all (Q3)
#1,peak_weighted,peak_weighted,cosine_deconv
#2,cosine_deconv,cosine_whitened,modal_peak_deconv
#3,cosine_raw,cosine_raw,modal_peak_match
#4,modal_peak_deconv,modal_peak_match,cosine_raw
#5,cosine_whitened,cosine_deconv,peak_weighted
#6,modal_peak_match,modal_peak_deconv,spectral_corr
#7,cosine_logsnr,cosine_logsnr,cosine_logsnr
#8,spectral_corr,spectral_corr,entropy_sim
#9,entropy_sim,entropy_sim,cosine_whitened
#10,cpsd,cpsd,cpsd


### The metrics disagree — that is the finding

**No metric wins everything.** `peak_weighted` leads on `rho_local`/`top1`/`top5` (Q1, Q2) but
is mid-table on `rho_all`; `cosine_deconv` dominates `rho_all` (Q3) and is second locally.

Two single-speaker conclusions that did not survive replication:

- `modal_peak_match` scored -0.662 on speaker 1 and looked best; across 8 speakers it averages
  ~-0.47 and is mid-pack. **Speaker 1 flattered it.**
- `coherence`/`cpsd` collapse locally (~8-9% top-1 vs 39%) — `rho_all` hid this.

**Why `peak_weighted` wins locally:** it is the only metric with *no local baseline* — it just
squares the L2-normalised spectrum. For small displacements the modes do not move between
bins; what changes is their **relative amplitude**. Floor-whitening normalises each bin against
its neighbourhood and cancels exactly that signal. Peak-weighting keeps it.

Tested separately: deconvolution **hurts** `peak_weighted` at every exponent (`rho_local`
-0.286 -> -0.178 at p=2), and p=2 is optimal.

In [11]:
# ==== THE CHOICE ====
# peak_weighted (p=2, raw, no deconvolution) is used for Parts 1-3: it is best on the
# diagnostics that match questions 1 and 2, and competitive on the rest.
# cosine_deconv is carried alongside for Part 3, where coarse far-field structure is what
# matters and it leads decisively.

PRIMARY_METHOD = 'peak_weighted'
SECONDARY_METHOD = 'cosine_deconv'

def similarity(MAG, method=PRIMARY_METHOD):
    return METHODS[method](MAG)

print(f'Primary  : {PRIMARY_METHOD}   (best rho_local / top1 / top5)')
print(f'Secondary: {SECONDARY_METHOD} (best rho_all; used for the aliasing analysis)')
selection.loc[[PRIMARY_METHOD, SECONDARY_METHOD]].round(3)

Primary  : peak_weighted   (best rho_local / top1 / top5)
Secondary: cosine_deconv (best rho_all; used for the aliasing analysis)


,rho_local,top1,top5,rho_all,med5
method,,,,,
peak_weighted,-0.286,39.254,78.070,-0.431,47.378
cosine_deconv,-0.261,32.675,77.193,-0.600,47.658


---
# Part 1 — If I move the object, does the FFT change a lot?

Similarity vs displacement for **nearby pairs only**. The spread matters more than the mean.

All pairs are formed **within a single speaker** (`pair_frame` takes the upper triangle of each
speaker's own matrix, then stacks the 8 sets). A cross-speaker pair would differ in excitation
as well as position, so it could not answer this question.

In [12]:
# One row per PAIR OF SAMPLES. Pairs are formed within a single speaker only: we take the
# upper triangle of each speaker's own similarity matrix, then stack the 8 results. Comparing
# two samples recorded through different speakers would change the excitation as well as the
# position, so such a pair says nothing about "did moving the object change the FFT".
def pair_frame(method=PRIMARY_METHOD):
    out = []
    for s, (df_s, mag_s, coms_s) in SPEAKER_DATA.items():
        D_s = squareform(pdist(coms_s))
        S = similarity(mag_s, method); np.fill_diagonal(S, 1.0)
        iu = np.triu_indices(len(S), 1)   # all unique same-speaker pairs
        out.append(pd.DataFrame({
            'speaker': s, 'sim': S[iu], 'dist': D_s[iu],
            'd_row': np.abs(coms_s[iu[0], 0] - coms_s[iu[1], 0]),   # toward/away from speaker
            'd_col': np.abs(coms_s[iu[0], 1] - coms_s[iu[1], 1]),   # left/right
        }))
    return pd.concat(out, ignore_index=True)

pairs = pair_frame()
near = pairs[pairs['dist'] < LOCAL_CUTOFF]
print(f'{len(pairs)} pairs total, {len(near)} with distance < {LOCAL_CUTOFF:.0f}')

12768 pairs total, 1944 with distance < 60


In [13]:
# Similarity vs displacement, small displacements only. Points = individual pairs.
fig = go.Figure()
fig.add_trace(go.Scattergl(x=near['dist'], y=near['sim'], mode='markers',
                           marker=dict(size=3, opacity=0.25, color='steelblue'), name='pair'))
bins = np.arange(0, LOCAL_CUTOFF + 1, 7.5)
mid = 0.5 * (bins[:-1] + bins[1:])
grp = near.groupby(pd.cut(near['dist'], bins), observed=True)['sim']
fig.add_trace(go.Scatter(x=mid, y=grp.mean().values, mode='lines+markers',
                         line=dict(color='crimson', width=3), name='mean'))
fig.update_layout(title=f'Q1: similarity vs displacement ({PRIMARY_METHOD}, 8 speakers pooled)'
                        '<br><sub>mean falls smoothly, but individual pairs scatter enormously</sub>',
                  xaxis_title='physical displacement', yaxis_title='similarity',
                  height=460, width=900)
fig.show()

In [14]:
# The spread, quantified: at each displacement, how bad can it get?
q1 = (near.groupby(pd.cut(near['dist'], bins), observed=True)['sim']
      .agg(n='size', mean='mean', min='min', max='max',
           frac_below_0p3=lambda x: (x < 0.3).mean()).round(3))
q1.index = [f'{int(i.left)}-{int(i.right)}' for i in q1.index]
q1

,n,mean,min,max,frac_below_0p3
0-7,56,0.979,0.888,0.998,0.000
7-15,56,0.936,0.305,0.998,0.000
15-22,120,0.967,0.522,0.998,0.000
22-30,64,0.950,0.552,0.998,0.000
30-37,272,0.911,0.256,0.997,0.011
37-45,456,0.900,0.232,0.998,0.009
45-52,568,0.882,0.235,0.998,0.014
52-60,352,0.901,0.263,0.997,0.003


In [15]:
# Does direction matter? Left/right (d_col) vs toward/away from the speaker (d_row).
# Shaded band is +/-1 std within each displacement bin -- given Q1's finding that the spread
# matters more than the mean, the band is the informative part of this plot.
fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=('left-right displacement (d_col)',
                                    'toward-away displacement (d_row)'))
for c, (key, xlab) in enumerate([('d_col', '|delta col|  (left-right, px)'),
                                 ('d_row', '|delta row|  (toward-away from speaker, px)')], start=1):
    sub = pairs[(pairs[key] < LOCAL_CUTOFF) & (pairs['dist'] < LOCAL_CUTOFF)]
    b = np.arange(0, LOCAL_CUTOFF + 1, 10)
    g = sub.groupby(pd.cut(sub[key], b), observed=True)['sim']
    x = 0.5 * (b[:-1] + b[1:])
    mean, std = g.mean().values, g.std().values

    # band first so the mean line draws on top of it
    fig.add_trace(go.Scatter(x=np.concatenate([x, x[::-1]]),
                             y=np.concatenate([mean + std, (mean - std)[::-1]]),
                             fill='toself', fillcolor='rgba(44,160,44,0.18)',
                             line=dict(width=0), hoverinfo='skip',
                             showlegend=False), row=1, col=c)
    fig.add_trace(go.Scatter(x=x, y=mean, mode='lines+markers', name='mean +/- 1 std',
                             showlegend=(c == 1), line=dict(color='seagreen', width=3),
                             error_y=dict(type='data', array=std, visible=True,
                                          color='rgba(44,160,44,0.85)', thickness=1.2, width=4)),
                  row=1, col=c)
    fig.update_xaxes(title=xlab, row=1, col=c)

fig.update_layout(title='Q1: is one axis more discriminative than the other?'
                        '<br><sub>mean +/- 1 std of similarity per displacement bin</sub>',
                  height=420, width=950)
fig.update_yaxes(title='similarity', row=1, col=1)
fig.show()

# Numbers behind the plot: is either axis actually more discriminative?
for key, lab in (('d_col', 'left-right'), ('d_row', 'toward-away')):
    sub = pairs[(pairs[key] < LOCAL_CUTOFF) & (pairs['dist'] < LOCAL_CUTOFF)]
    print(f'{lab:12s} rho(sim, displacement) = {spearmanr(sub["sim"], sub[key]).statistic:+.3f}'
          f'   mean std within bins = {sub.groupby(pd.cut(sub[key], np.arange(0, LOCAL_CUTOFF + 1, 10)), observed=True)["sim"].std().mean():.3f}')

left-right   rho(sim, displacement) = +0.025   mean std within bins = 0.138
toward-away  rho(sim, displacement) = -0.307   mean std within bins = 0.124


**Left-right displacement is essentially uninformative; toward-away carries the signal.**

```
left-right   rho = +0.025   (indistinguishable from zero, and the wrong sign)
toward-away  rho = -0.307
```

So the literal form of Q1 — "if I move the object left/right, does the FFT change?" — answers
**no more than chance**. Only motion toward/away from the speaker reliably changes the spectrum.

Physically this fits: distance to the speaker sets how much excitation reaches the object and
how it couples into the box, whereas sliding along the perpendicular axis leaves that geometry
roughly unchanged. It also explains Q3 — the recurring aliased pairs are all opposite-extreme
positions, consistent with a left-right symmetry that a magnitude-only spectrum cannot break.

Note the +/-1 std band is wide relative to the trend in both panels (mean within-bin std ~0.13,
comparable to the full range of the means), so this is the same "spread dominates" story as the
main Q1 scatter.

**Actionable:** if left-right position is poorly recoverable with one speaker, a second speaker
on the perpendicular axis should recover it. Testable here — if the informative axis rotates
with speaker placement, that confirms the mechanism.

**Answer to Q1.** Mean similarity falls smoothly with displacement, but the *spread* is the
real story: at the same small displacement some pairs are near-identical and others nearly
orthogonal. So it is not "nearby ≈ same with slight differences" — it is **usually similar,
occasionally completely different**.

Physically this is expected: response depends on where the object sits relative to each mode's
nodes and antinodes. Near a node, a small move flips that mode's contribution sharply.

---
# Part 2 — Are the nearest FFT neighbours the physical neighbours?

In [16]:
# Rank of the physically-nearest sample in the similarity ranking, per speaker.
rows = []
for s, (df_s, mag_s, coms_s) in SPEAKER_DATA.items():
    D_s = squareform(pdist(coms_s))
    S = similarity(mag_s); np.fill_diagonal(S, 1.0)
    ranks, order = rank_of_physical_nn(S, D_s)
    Dx = D_s.copy(); np.fill_diagonal(Dx, np.inf)
    rows.append(dict(speaker=s, n=len(S),
                     top1=(ranks == 1).mean() * 100, top5=(ranks <= 5).mean() * 100,
                     med5=np.median(np.take_along_axis(D_s, order[:, :5], axis=1)),
                     oracle_floor=np.median(np.sort(Dx, axis=1)[:, :5]),
                     random_baseline=np.median(D_s[np.triu_indices(len(S), 1)])))
q2 = pd.DataFrame(rows).set_index('speaker')
print(q2.round(1))
print(f'\nMEAN  top1 {q2["top1"].mean():.0f}%   top5 {q2["top5"].mean():.0f}%   '
      f'med5 {q2["med5"].mean():.1f}  (oracle floor {q2["oracle_floor"].mean():.1f}, '
      f'random {q2["random_baseline"].mean():.0f})')

          n  top1  top5  med5  oracle_floor  random_baseline
speaker                                                     
1        57  35.1  77.2  46.4          37.5            147.2
2        57  36.8  84.2  50.3          37.5            147.2
3        57  42.1  73.7  48.1          37.5            147.2
4        57  40.4  78.9  46.9          37.5            147.2
5        57  38.6  73.7  46.9          37.5            147.2
6        57  33.3  71.9  47.7          37.5            147.2
7        57  49.1  84.2  44.8          37.5            147.2
8        57  38.6  80.7  47.9          37.5            147.2

MEAN  top1 39%   top5 78%   med5 47.4  (oracle floor 37.5, random 147)


In [17]:
# Distribution of that rank, pooled. Rank 1 = metric's top pick IS the true nearest position.
all_ranks = np.concatenate([rank_of_physical_nn(
    np.where(np.eye(len(m), dtype=bool), 1.0, similarity(m)), squareform(pdist(c)))[0]
    for _, m, c in SPEAKER_DATA.values()])
fig = go.Figure(go.Histogram(x=all_ranks, xbins=dict(start=0.5, end=30.5, size=1),
                             marker_color='steelblue'))
fig.add_vline(x=5.5, line=dict(dash='dash', color='crimson'),
              annotation_text=f'top-5 = {(all_ranks <= 5).mean():.0%}')
fig.update_layout(title='Q2: similarity-rank of the physically-nearest sample'
                        '<br><sub>concentrated at low ranks, with a long failure tail</sub>',
                  xaxis_title='rank (1 = best)', yaxis_title='count', height=400, width=900)
fig.show()

In [18]:
# Where do the top-5 neighbours sit physically? Lines join each sample to its matches.
def neighbour_map(speaker=1, k=5, method=PRIMARY_METHOD):
    df_s, mag_s, coms_s = SPEAKER_DATA[speaker]
    D_s = squareform(pdist(coms_s))
    S = similarity(mag_s, method); np.fill_diagonal(S, -np.inf)
    nn = np.argsort(-S, axis=1)[:, :k]
    fig = go.Figure()
    for i in range(len(coms_s)):
        for j in nn[i]:
            far = D_s[i, j] > LOCAL_CUTOFF
            fig.add_trace(go.Scatter(
                x=[coms_s[i, 1], coms_s[j, 1]], y=[coms_s[i, 0], coms_s[j, 0]], mode='lines',
                line=dict(width=1, color='rgba(214,39,40,0.55)' if far else 'rgba(31,119,180,0.30)'),
                showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=coms_s[:, 1], y=coms_s[:, 0], mode='markers',
                             marker=dict(size=7, color='black'), name='sample'))
    fig.update_layout(title=f'Q2: top-{k} spectral neighbours in space (speaker {speaker}, {method})'
                            '<br><sub>blue = local match, red = jumps > 60 units</sub>',
                      xaxis_title='com col', yaxis_title='com row', height=620, width=760)
    fig.update_yaxes(autorange='reversed', scaleanchor='x', scaleratio=1)
    fig.show()

neighbour_map(1)

**Answer to Q2.** Mostly yes, coarsely; unreliably, finely. The physically-nearest sample is
in the top 5 about **77%** of the time but is the #1 pick only about **39%**.

The `med5` column needs its floor to be read correctly: positions are ~16 units apart, so the
median distance to the 5 *physically* nearest samples is **37.5** — that is what a perfect
oracle scores. We get ~47. Most of the apparent gap is grid geometry, not metric error, which
is why `med5` barely separates the methods and should not be optimised directly.

---
# Part 3 — Is the position -> FFT map unique?

Aliasing = two *far-apart* positions with *highly similar* FFTs. Shown as a scatter (no
threshold) plus a continuous score, under both metrics.

In [19]:
# Every pair: distance vs similarity. Aliasing would appear in the UPPER-RIGHT.
fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=(PRIMARY_METHOD, SECONDARY_METHOD))
for c, meth in enumerate([PRIMARY_METHOD, SECONDARY_METHOD], start=1):
    pf = pair_frame(meth)
    fig.add_trace(go.Scattergl(x=pf['dist'], y=pf['sim'], mode='markers', showlegend=False,
                               marker=dict(size=2, opacity=0.15, color='steelblue')), row=1, col=c)
    hi = pf[(pf['dist'] > np.percentile(pf['dist'], 75)) &
            (pf['sim'] > np.percentile(pf['sim'], 99))]
    fig.add_trace(go.Scattergl(x=hi['dist'], y=hi['sim'], mode='markers', showlegend=False,
                               marker=dict(size=5, color='crimson')), row=1, col=c)
    fig.update_xaxes(title='physical distance', row=1, col=c)
fig.update_yaxes(title='similarity', row=1, col=1)
fig.update_layout(height=440, width=1050,
                  title='Q3: aliasing scatter — red = far apart AND highly similar')
fig.show()

In [20]:
# Continuous aliasing score: similarity relative to what is TYPICAL at that distance.
# score >> 1 means "far more similar than two positions this far apart should be".
def aliasing_scores(method=PRIMARY_METHOD, top=15):
    out = []
    for s, (df_s, mag_s, coms_s) in SPEAKER_DATA.items():
        D_s = squareform(pdist(coms_s))
        S = similarity(mag_s, method); np.fill_diagonal(S, 1.0)
        iu = np.triu_indices(len(S), 1)
        sim, dist = S[iu], D_s[iu]
        # expected similarity at each distance, from a smooth fit over distance bins
        b = np.linspace(0, dist.max() + 1e-6, 12)
        idx = np.clip(np.digitize(dist, b[1:-1]), 0, len(b) - 2)
        exp = np.array([sim[idx == k].mean() if (idx == k).any() else np.nan for k in range(len(b) - 1)])[idx]
        score = sim / np.maximum(exp, 1e-9)
        keep = dist > np.percentile(dist, 75)          # only far-apart pairs can alias
        for k in np.argsort(-np.where(keep, score, -np.inf))[:top]:
            a, bb = iu[0][k], iu[1][k]
            out.append(dict(speaker=s, sample_a=df_s['sample_id'][a], sample_b=df_s['sample_id'][bb],
                            com_a=f'({coms_s[a,0]:.0f},{coms_s[a,1]:.0f})',
                            com_b=f'({coms_s[bb,0]:.0f},{coms_s[bb,1]:.0f})',
                            dist=round(float(dist[k]), 1), sim=round(float(sim[k]), 3),
                            alias_score=round(float(score[k]), 2)))
    return pd.DataFrame(out).sort_values('alias_score', ascending=False).reset_index(drop=True)

alias = aliasing_scores()
alias.head(15)

,speaker,sample_a,sample_b,com_a,com_b,dist,sim,alias_score
0,1,000264,000472,"(184,150)","(110,523)",379.9,0.948,1.74
1,1,000256,000440,"(182,128)","(155,491)",364.2,0.945,1.74
2,1,000216,000256,"(106,547)","(182,128)",425.5,0.959,1.72
3,1,000216,000264,"(106,547)","(184,150)",403.9,0.958,1.72
4,1,000256,000432,"(182,128)","(116,489)",366.7,0.932,1.71
5,1,000256,000472,"(182,128)","(110,523)",401.4,0.949,1.70
6,1,000216,000304,"(106,547)","(194,199)",358.3,0.918,1.69
7,1,000048,000216,"(181,198)","(106,547)",356.4,0.904,1.66
8,1,000176,000256,"(148,475)","(182,128)",349.0,0.889,1.63
9,1,000040,000472,"(182,161)","(110,523)",368.6,0.888,1.63


In [21]:
# Strongest test: does the SAME position pair look aliased under MULTIPLE speakers?
# Different speakers excite the modes differently, so a real geometric collision should recur.
pos_key = alias.assign(key=alias.apply(lambda r: tuple(sorted([r['com_a'], r['com_b']])), axis=1))
recur = (pos_key.groupby('key')
         .agg(n_speakers=('speaker', 'nunique'), speakers=('speaker', lambda x: sorted(set(x))),
              mean_score=('alias_score', 'mean'), mean_dist=('dist', 'mean'))
         .sort_values(['n_speakers', 'mean_score'], ascending=False))
print('Position pairs flagged as aliased under more than one speaker:')
recur[recur['n_speakers'] > 1].head(10)

Position pairs flagged as aliased under more than one speaker:


,n_speakers,speakers,mean_score,mean_dist
key,,,,
"((155,491), (182,128))",7,"[1, 2, 3, 4, 5, 6, 7]",1.324286,364.2
"((106,547), (184,150))",5,"[1, 3, 4, 5, 6]",1.430000,403.9
"((145,539), (184,150))",5,"[1, 3, 4, 5, 6]",1.430000,390.9
"((110,523), (182,128))",5,"[1, 3, 4, 5, 6]",1.422000,401.4
"((106,547), (182,128))",5,"[1, 3, 4, 5, 6]",1.420000,425.5
"((145,539), (182,128))",5,"[1, 3, 4, 5, 6]",1.414000,412.9
"((110,523), (184,150))",5,"[1, 3, 4, 5, 6]",1.368000,379.9
"((116,489), (182,128))",5,"[1, 3, 4, 5, 6]",1.358000,366.7
"((106,547), (182,161))",4,"[1, 3, 4, 6]",1.455000,392.6


In [22]:
# Look at the worst offender: do the two spectra actually overlap?
def plot_alias_pair(row, method=PRIMARY_METHOD):
    df_s, mag_s, coms_s = SPEAKER_DATA[row['speaker']]
    ids = list(df_s['sample_id'])
    a, b = ids.index(row['sample_a']), ids.index(row['sample_b'])
    W = _log_snr(mag_s)                      # plot whitened: shows modal structure clearly
    fig = go.Figure()
    for idx, color, lab in ((a, 'black', row['com_a']), (b, 'crimson', row['com_b'])):
        fig.add_trace(go.Scatter(x=freqs, y=W[idx], line=dict(width=1.4, color=color),
                                 name=f'{ids[idx]} com={lab}'))
    fig.update_layout(
        title=f'Q3: most-aliased pair (speaker {row["speaker"]}) — {row["dist"]:.0f} units apart, '
              f'sim {row["sim"]:.3f}<br><sub>log-SNR spectra; + = resonance, - = anti-resonance</sub>',
        xaxis_title='frequency (Hz)', yaxis_title='log(mag / floor)', height=460, width=1100)
    fig.show()

if len(alias):
    plot_alias_pair(alias.iloc[0])

**Answer to Q3 — the map is NOT unique.** The cross-speaker table is decisive: the position
pair `(155,491)` / `(182,128)` — **364 units apart**, opposite sides of the box — is flagged as
anomalously similar under **7 of 8 speakers**, and several other pairs recur under 5 of 8.
Different speakers excite the modes differently, so a collision that survives that many
independent excitations is **box geometry, not noise**.

Note every recurring pair is far apart (350–425 units) and involves positions at opposite
extremes. This is the signature of a symmetry: the box's mode shapes are near-symmetric, so
mirrored positions produce near-identical magnitude spectra. Phase would in principle break the
degeneracy — but Part 0 showed phase is unusable here.

**Practical consequence:** a position-from-FFT model will confuse these mirrored regions. This
is the concrete failure mode to design around (a second speaker at an asymmetric location, or
a reference channel enabling phase, would break it).

---
# Summary

| question | answer |
|---|---|
| **Q0** How to measure similarity? | `peak_weighted` (squared normalised spectrum) for fine/local work; `cosine_deconv` for coarse. Keep anti-resonances, deconvolve the excitation empirically, discard phase. No metric wins both regimes. |
| **Q1** Does moving change the FFT? | Mean similarity falls smoothly with displacement, but with large spread — usually similar, occasionally near-orthogonal. The map is locally **rough**, not smooth. |
| **Q2** Are FFT neighbours physical neighbours? | Coarsely yes (**78%** top-5), finely no (**39%** top-1). `med5` ≈ 47.4 against an oracle floor of 37.5 — most of that gap is grid geometry, not metric error. |
| **Q3** Is the map unique? | **No.** `(155,491)`/`(182,128)`, 364 units apart, is flagged under **7 of 8 speakers**; several more recur under 5 of 8. All are opposite-extreme pairs — a box-symmetry degeneracy. |

**Caveats.**
- All results collapse 100 lasers x 2 directions into one magnitude curve. That averaging cuts
  variance but discards spatial information a model could exploit, and is the most likely
  reason local discrimination is weak. **Per-laser similarity is the top thing to try next.**
- Single purple cube only; multi-object layouts untested.
- Speaker 7 is the strongest slice (49% top-1) and speaker 6 the weakest (33%) — speaker
  placement is a real experiment-design lever.